# NB10 Pilot — LogitDistLoss Sanity Check

**Purpose:** Quick diagnostic pilot, NOT a thesis-numbered notebook. Before committing to the full 30-run marathon re-fit (~25-37 hours) with `elementwise_loss="LogitDistLoss()"`, this notebook runs a single, short PySR fit to confirm:

1. PySR accepts and runs cleanly with `elementwise_loss="LogitDistLoss()"` on this codebase/environment.
2. Applying `sigmoid()` to the raw expression output produces genuinely bounded (0,1) probabilities, with no clipping needed anywhere.
3. An early read on whether the resulting expression looks structurally sane/interpretable, before spending the full marathon budget.

**This is explicitly a throwaway diagnostic.** Reduced `niterations`/`populations` relative to the canonical marathon config — the resulting formula and metrics are NOT meant to be quoted anywhere in the thesis. Output goes to `results/gp_runs_pilot/`, fully separate from both the archived `*_v1_mse` results and the fresh canonical `results/gp_runs/`.

**Inputs:** `data/processed/features_curated.parquet`, `data/processed/feature_config.json` (GP_TERMINALS), `data/processed/split_random.csv` (train/val/test labels).

**Output:** `results/gp_runs_pilot/pilot_model.pkl`, printed diagnostics only — no CSVs/figures meant for the thesis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json, pickle, time, sys, warnings
warnings.filterwarnings("ignore")

_nb_dir   = Path().resolve()
PROJECT   = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
DATA_PROC = PROJECT / "data" / "processed"
PILOT_DIR = PROJECT / "results" / "gp_runs_pilot"
PILOT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.metrics import compute_ece, compute_metrics

feat = pd.read_parquet(DATA_PROC / "features_curated.parquet")
with open(DATA_PROC / "feature_config.json") as f:
    cfg = json.load(f)
GP_TERMINALS = cfg["GP_TERMINALS"]

split_df = pd.read_csv(DATA_PROC / "split_random.csv")
feat_split = feat.merge(split_df[["patientunitstayid", "split"]], on="patientunitstayid")

train_df = feat_split[feat_split["split"] == "train"].reset_index(drop=True)
val_df   = feat_split[feat_split["split"] == "val"].reset_index(drop=True)
test_df  = feat_split[feat_split["split"] == "test"].reset_index(drop=True)

X_train_gp = train_df[GP_TERMINALS].copy()
y_train    = train_df["hospital_mortality"].to_numpy(dtype=np.float64)
X_val_gp   = val_df[GP_TERMINALS].copy()
y_val      = val_df["hospital_mortality"].to_numpy(dtype=np.float64)
X_test_gp  = test_df[GP_TERMINALS].copy()
y_test     = test_df["hospital_mortality"].to_numpy(dtype=np.float64)

print(f"Train: {len(train_df):,} ({y_train.mean()*100:.2f}%)  "
      f"Val: {len(val_df):,} ({y_val.mean()*100:.2f}%)  "
      f"Test: {len(test_df):,} ({y_test.mean()*100:.2f}%)")
print(f"GP_TERMINALS: {len(GP_TERMINALS)} features")
print(f"Pilot output dir: {PILOT_DIR}")

In [ ]:
print("Importing PySR -- Julia compilation may take 5-10 min on first run ...")
t_import = time.time()
from pysr import PySRRegressor
print(f"PySR import complete in {time.time()-t_import:.0f}s.")

In [ ]:
# ============================================================================
# PILOT FIT -- reduced niterations/populations vs the canonical marathon
# (niterations=6000, populations=30). This is a diagnostic, not a result.
# elementwise_loss="LogitDistLoss()" treats the expression output as a logit;
# PySR internally applies sigmoid + binary cross-entropy during the search.
# We additionally apply sigmoid ourselves post-hoc to get probabilities for
# our own metric computation (compute_ece/compute_metrics expect probabilities).
# ============================================================================
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

gp_pilot = PySRRegressor(
    niterations      = 400,
    populations      = 15,
    maxsize          = 24,
    parsimony        = 0.001,
    binary_operators = ["+", "-", "*", "/", "max", "min"],
    unary_operators  = ["log", "sqrt", "exp", "abs"],
    elementwise_loss  = "LogitDistLoss()",
    model_selection  = "best",
    random_state     = 42,
    verbosity        = 0,
    progress         = False,
    output_directory = str(PILOT_DIR),
)

print("Starting pilot PySR fit (niterations=400, populations=15) ...")
t0 = time.time()
gp_pilot.fit(X_train_gp, y_train)
elapsed = (time.time() - t0) / 60
print(f"Pilot fit complete in {elapsed:.1f} min.")

with open(PILOT_DIR / "pilot_model.pkl", "wb") as f:
    pickle.dump(gp_pilot, f)
print(f"Saved: {PILOT_DIR / 'pilot_model.pkl'}")

In [ ]:
# ============================================================================
# Evaluate the Pareto front on validation (raw logit -> sigmoid -> metrics),
# select best by the canonical fitness (AUROC - 0.5*ECE), then check test set.
# Key sanity checks: (1) does sigmoid output ever need clipping? (2) does the
# selected formula look structurally interpretable?
# ============================================================================
pareto_rows = []
for idx, eq_row in gp_pilot.equations_.iterrows():
    try:
        raw_val = gp_pilot.predict(X_val_gp, index=idx)
        raw_val = np.where(np.isfinite(raw_val), raw_val, 0.0)
        prob_val = sigmoid(raw_val)
        m = compute_metrics(y_val, prob_val, label=f"c{int(eq_row['complexity'])}",
                             fitness_alpha=1.0, fitness_beta=0.5)
        m["complexity"] = int(eq_row["complexity"])
        m["equation"]   = str(eq_row["equation"])
        m["eq_idx"]     = idx
        m["raw_min"]    = float(raw_val.min())
        m["raw_max"]    = float(raw_val.max())
        m["prob_min"]   = float(prob_val.min())
        m["prob_max"]   = float(prob_val.max())
        pareto_rows.append(m)
    except Exception as e:
        print(f"  c={eq_row['complexity']} skipped: {e}")

pareto_df = pd.DataFrame(pareto_rows)
print("Pareto front (validation set, sigmoid-transformed probabilities):")
print(pareto_df[["complexity", "auroc", "ece_10bin", "brier", "fitness",
                 "raw_min", "raw_max", "prob_min", "prob_max"]].to_string(index=False))

best_idx_row = pareto_df["fitness"].idxmax()
best = pareto_df.loc[best_idx_row]
eq_idx = int(best["eq_idx"])

print(f"\nSelected by val fitness: complexity={int(best['complexity'])}")
print(f"  Equation: {best['equation']}")
print(f"  Raw (logit) range on val: [{best['raw_min']:.4f}, {best['raw_max']:.4f}]")
print(f"  Sigmoid (prob) range on val: [{best['prob_min']:.6f}, {best['prob_max']:.6f}]")

# -- Test set evaluation --
raw_test  = gp_pilot.predict(X_test_gp, index=eq_idx)
raw_test  = np.where(np.isfinite(raw_test), raw_test, 0.0)
prob_test = sigmoid(raw_test)
test_m = compute_metrics(y_test, prob_test, label="pilot_test")

print(f"\nTest set (n={len(y_test):,}):")
print(f"  AUROC: {test_m['auroc']:.4f}  ECE: {test_m['ece_10bin']:.4f}  Brier: {test_m['brier']:.4f}")
print(f"  Raw (logit) range on test: [{raw_test.min():.4f}, {raw_test.max():.4f}]")
print(f"  Sigmoid (prob) range on test: [{prob_test.min():.6f}, {prob_test.max():.6f}]")
print(f"  No clipping applied anywhere -- sigmoid output is bounded by construction.")

# -- Loose comparison vs the archived MSE-loss canonical result, for context --
print("\nFor context -- archived MSE-loss canonical result (c=23, seed=5, full 6000-iter marathon):")
print("  Test AUROC=0.7432, ECE=0.0162, Brier=0.1204 (NOT a fair comparison -- pilot used far fewer")
print("  iterations and a different loss; only useful as a rough sanity-check reference point.)")

## Full-budget single-seed run

The reduced-budget pilot above (niterations=400, populations=15) ran cleanly and confirmed sigmoid outputs are genuinely bounded — but it showed poor calibration (ECE=0.372), with predicted probabilities clustered in [0.48, 0.78], well above the true ~16.9% base rate. This looks like an intercept/scale problem rather than a structural one: to predict near the true base rate the model needs a logit around `logit(0.169) ≈ -1.59`, but the pilot's logit range never got there — plausibly because 400 iterations isn't enough search budget to find the right large constant offset in logit space (the original MSE-loss marathon itself needed the full 6000 iterations to converge, per CLAUDE.md).

This cell runs **one single seed at the full canonical budget** (niterations=6000, populations=30, same operators/maxsize/parsimony as the original marathon, only the loss function changed) to test whether calibration recovers once the search has its intended budget. Expected runtime: ~50-75 minutes, matching the original marathon's per-seed timing. Output goes to `results/gp_runs_pilot_fullbudget/`, separate from everything else.

In [5]:
# ============================================================================
# FULL-BUDGET single-seed fit, LogitDistLoss, canonical marathon settings.
# This is still a diagnostic (one seed, not the 30-run marathon), but at the
# intended search budget -- directly tests whether the earlier pilot's poor
# calibration was a budget artifact or a real problem with this approach.
# ============================================================================
FULLBUDGET_DIR = PROJECT / "results" / "gp_runs_pilot_fullbudget"
FULLBUDGET_DIR.mkdir(parents=True, exist_ok=True)

gp_full = PySRRegressor(
    niterations      = 6000,
    populations      = 30,
    maxsize          = 24,
    parsimony        = 0.001,
    binary_operators = ["+", "-", "*", "/", "max", "min"],
    unary_operators  = ["log", "sqrt", "exp", "abs"],
    elementwise_loss  = "LogitDistLoss()",
    model_selection  = "best",
    random_state     = 42,
    verbosity        = 0,
    progress         = False,
    output_directory = str(FULLBUDGET_DIR),
)

print("Starting full-budget single-seed PySR fit (niterations=6000, populations=30, LogitDistLoss) ...")
print("Expected runtime: ~50-75 minutes based on the original marathon's per-seed timings.")
t0 = time.time()
gp_full.fit(X_train_gp, y_train)
elapsed = (time.time() - t0) / 60
print(f"Full-budget fit complete in {elapsed:.1f} min.")

with open(FULLBUDGET_DIR / "fullbudget_model.pkl", "wb") as f:
    pickle.dump(gp_full, f)
print(f"Saved: {FULLBUDGET_DIR / 'fullbudget_model.pkl'}")

Starting full-budget single-seed PySR fit (niterations=6000, populations=30, LogitDistLoss) ...
Expected runtime: ~50-75 minutes based on the original marathon's per-seed timings.
Full-budget fit complete in 111.1 min.
Saved: C:\ML PROJECT\sepsis-gp\results\gp_runs_pilot_fullbudget\fullbudget_model.pkl


In [6]:
# ============================================================================
# Evaluate the full-budget run's Pareto front on validation, select by the
# canonical fitness, check test set, and compare directly against both the
# reduced-budget pilot and the archived MSE-loss canonical result.
# ============================================================================
pareto_rows_full = []
for idx, eq_row in gp_full.equations_.iterrows():
    try:
        raw_val = gp_full.predict(X_val_gp, index=idx)
        raw_val = np.where(np.isfinite(raw_val), raw_val, 0.0)
        prob_val = sigmoid(raw_val)
        m = compute_metrics(y_val, prob_val, label=f"c{int(eq_row['complexity'])}",
                             fitness_alpha=1.0, fitness_beta=0.5)
        m["complexity"] = int(eq_row["complexity"])
        m["equation"]   = str(eq_row["equation"])
        m["eq_idx"]     = idx
        m["raw_min"]    = float(raw_val.min())
        m["raw_max"]    = float(raw_val.max())
        m["prob_min"]   = float(prob_val.min())
        m["prob_max"]   = float(prob_val.max())
        pareto_rows_full.append(m)
    except Exception as e:
        print(f"  c={eq_row['complexity']} skipped: {e}")

pareto_df_full = pd.DataFrame(pareto_rows_full)
print("Pareto front (validation set, full-budget LogitDistLoss run):")
print(pareto_df_full[["complexity", "auroc", "ece_10bin", "brier", "fitness",
                       "raw_min", "raw_max", "prob_min", "prob_max"]].to_string(index=False))

best_idx_row_full = pareto_df_full["fitness"].idxmax()
best_full = pareto_df_full.loc[best_idx_row_full]
eq_idx_full = int(best_full["eq_idx"])

print(f"\nSelected by val fitness: complexity={int(best_full['complexity'])}")
print(f"  Equation: {best_full['equation']}")
print(f"  Raw (logit) range on val: [{best_full['raw_min']:.4f}, {best_full['raw_max']:.4f}]")
print(f"  Sigmoid (prob) range on val: [{best_full['prob_min']:.6f}, {best_full['prob_max']:.6f}]")

raw_test_full  = gp_full.predict(X_test_gp, index=eq_idx_full)
raw_test_full  = np.where(np.isfinite(raw_test_full), raw_test_full, 0.0)
prob_test_full = sigmoid(raw_test_full)
test_m_full = compute_metrics(y_test, prob_test_full, label="fullbudget_test")

print(f"\nTest set (n={len(y_test):,}):")
print(f"  AUROC: {test_m_full['auroc']:.4f}  ECE: {test_m_full['ece_10bin']:.4f}  Brier: {test_m_full['brier']:.4f}")
print(f"  Raw (logit) range on test: [{raw_test_full.min():.4f}, {raw_test_full.max():.4f}]")
print(f"  Sigmoid (prob) range on test: [{prob_test_full.min():.6f}, {prob_test_full.max():.6f}]")

print("\nComparison:")
print(f"  Pilot (400 iter):        AUROC=0.7270  ECE=0.3720  Brier=0.2704  logit range [-0.08, 1.27]")
print(f"  Full-budget (6000 iter): AUROC={test_m_full['auroc']:.4f}  ECE={test_m_full['ece_10bin']:.4f}  "
      f"Brier={test_m_full['brier']:.4f}  logit range [{raw_test_full.min():.2f}, {raw_test_full.max():.2f}]")
print(f"  Archived MSE canonical:  AUROC=0.7432  ECE=0.0162  Brier=0.1204")
print(f"\n  Target logit for true base rate (16.9%): logit(0.169) = {np.log(0.169/0.831):.3f}")

Pareto front (validation set, full-budget LogitDistLoss run):
 complexity  auroc  ece_10bin  brier  fitness   raw_min  raw_max  prob_min  prob_max
          1 0.5000     0.3711 0.2783   0.3145  0.161492 0.161492  0.540286  0.540286
          3 0.6480     0.3649 0.2677   0.4655  0.015978 1.278272  0.503995  0.782155
          4 0.6873     0.3810 0.2772   0.4968  0.063895 0.609123  0.515968  0.647741
          5 0.6806     0.3673 0.2692   0.4969  0.015162 1.212926  0.503790  0.770816
          6 0.6720     0.3729 0.2730   0.4855  0.053924 1.108268  0.513478  0.751806
          7 0.6952     0.3701 0.2707   0.5102  0.022650 1.019265  0.505662  0.734829
          8 0.6986     0.3726 0.2731   0.5123  0.027278 0.926028  0.506819  0.716269
          9 0.7056     0.3713 0.2713   0.5200  0.021917 1.025463  0.505479  0.736035
         10 0.7189     0.3721 0.2714   0.5328  0.025918 1.011454  0.506479  0.733305
         11 0.7246     0.3728 0.2731   0.5382  0.000000 1.135447  0.500000  0.756843
   

## Findings

*[To be completed once Cells 1-4 have been run. Record: (1) total wall-clock time including Julia import/compile, (2) whether sigmoid outputs were genuinely bounded with no clipping needed, (3) the selected pilot equation's structure/complexity and whether it looks clinically interpretable, (4) pilot test AUROC/ECE/Brier as a rough sanity check only -- NOT comparable to the canonical marathon given the reduced niterations/populations, (5) a go/no-go recommendation on proceeding to the full 30-run LogitDistLoss marathon.]*